# HGRIA - Hand Gesture Recognition for Interactive Applications
## Google Colab Launch Notebook

```
┌─────────────────────────────────────────────────────────────┐
│                    ARCHITECTURE (COLAB)                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Browser (Frontend)     ngrok Tunnel      Colab Backend   │
│   ┌──────────────┐      ┌──────────┐     ┌──────────────┐   │
│   │  GitHub Pages │ ←─── │  HTTPS   │ ←── │  Flask +     │   │
│   │  / Vercel    │      │  Tunnel  │     │  MediaPipe   │   │
│   └──────────────┘      └──────────┘     └──────────────┘   │
│         │                                          │        │
│         │           Google Drive                    │        │
│         └──────────────┬───────────────────────────┘        │
│                        │ logs/                             │
└─────────────────────────────────────────────────────────────┘
```

### Prerequisites
- Google Account with Google Drive access
- HGRIA project saved in Google Drive at `/MyDrive/HGRIA/`
- ngrok account (free tier works)
- WebRTC-compatible browser (Chrome, Edge, Firefox)

### How it works
1. Project is copied from Google Drive to Colab runtime
2. Backend runs Flask server on Colab with MediaPipe
3. ngrok creates HTTPS tunnel to expose backend
4. Frontend connects via WebSocket and sends webcam frames
5. Backend processes frames and sends gesture commands back

In [ ]:
# Step 1: Install pinned dependencies
# Run this first — Colab may have mismatched versions pre-installed
!pip install --no-cache-dir \
    "numpy==1.26.4" \
    "tensorflow==2.18.0" \
    "protobuf==4.25.3" \
    "mediapipe==0.10.21" \
    "opencv-contrib-python==4.11.0.86"

In [ ]:
# Step 2: Verify installed versions
import numpy as np
import google.protobuf
import mediapipe as mp
import tensorflow as tf

print("NumPy:", np.__version__)
print("Protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)
print("MediaPipe OK")
print("TensorFlow OK")

In [ ]:
# Step 3: Mount Google Drive and verify project files
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

SOURCE_PATH = '/content/drive/MyDrive/HGRIA'
REQUIREMENTS_FILE = os.path.join(SOURCE_PATH, 'requirements.txt')

if not os.path.exists(REQUIREMENTS_FILE):
    raise FileNotFoundError(
        f"requirements.txt not found at {REQUIREMENTS_FILE}.\n"
        "Please ensure HGRIA project is saved in your Google Drive at:\n"
        "  /content/drive/MyDrive/HGRIA/"
    )

print(f"✓ Project files found at {SOURCE_PATH}")

In [ ]:
# Step 4: Copy project to Colab runtime and install dependencies
import shutil
import subprocess
import sys

DEST = '/content/HGRIA'

def _ignore_pycache(src, names):
    """Exclude __pycache__ directories and .pyc files from the copy."""
    return [n for n in names if n == '__pycache__' or n.endswith('.pyc')]

if os.path.exists(DEST):
    shutil.rmtree(DEST)
    print(f"Removed stale copy at {DEST}")

shutil.copytree(SOURCE_PATH, DEST, ignore=_ignore_pycache)
print(f"\u2713 Copied project to {DEST} (without __pycache__)")

sys.path.insert(0, DEST)
os.chdir(DEST)

result = subprocess.run(
    ['pip', 'install', '-q', '-r', 'requirements.txt'],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"Failed to install dependencies:\n{result.stderr}"
    )

print("\u2713 Dependencies installed successfully")

In [ ]:
# Step 5: Install pyngrok and configure ngrok authentication
#
# ── EDIT THIS ──────────────────────────────────────────────────────────────
# Paste your authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken
# Leave as empty string to use an anonymous tunnel (disconnects after ~2 h).
NGROK_AUTHTOKEN = ""  # e.g. "2abc123XYZ_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
# ───────────────────────────────────────────────────────────────────────────

import re
import subprocess

subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
from pyngrok import ngrok

token = NGROK_AUTHTOKEN.strip()
if token:
    # Basic sanity check: ngrok tokens are alphanumeric/underscore/dash, 20+ chars
    if not re.match(r'^[A-Za-z0-9_\-]{20,}$', token):
        raise ValueError(
            "NGROK_AUTHTOKEN does not look valid.\n"
            "Expected an alphanumeric string of 20+ characters.\n"
            "Get yours from: https://dashboard.ngrok.com/get-started/your-authtoken"
        )
    ngrok.set_auth_token(token)
    print("\u2713 ngrok authenticated — tunnel will open when the server starts (Step 7)")
else:
    print("\u26a0 No authtoken set — anonymous tunnel (may disconnect after ~2 h)")
    print("  Paste your token into NGROK_AUTHTOKEN above to avoid this")

In [ ]:
# Step 6: Patch config for Colab environment
import json
from pathlib import Path

PROJECT_ROOT = Path(DEST)
config_path = str(PROJECT_ROOT / 'config' / 'config.json')

with open(config_path, 'r') as f:
    config = json.load(f)

config['camera']['colab_mode'] = True
config['server']['cors_origins'] = '*'
config['logging']['log_to_file'] = True
config['logging']['log_file_path'] = '/content/drive/MyDrive/HGRIA/logs/'

# Ensure logs directory exists on Drive
os.makedirs(config['logging']['log_file_path'], exist_ok=True)

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("\u2713 Config patched:")
print(f"  - colab_mode   : {config['camera']['colab_mode']}")
print(f"  - cors_origins : {config['server']['cors_origins']}")
print(f"  - log_file_path: {config['logging']['log_file_path']}")

In [ ]:
# Step 6b: Clear Python module cache and apply in-place patches
#
# This step does two things:
#
# 1. Patches configuration.py directly in the runtime copy so it works even
#    when the Google Drive copy has not been updated yet.  The patch fixes
#    ConfigurationManager.__getattr__ to return gesture_cooldowns_ms as a
#    plain dict instead of wrapping it in _Namespace (which has no .get()).
#
# 2. Evicts all backend.* modules from sys.modules so the next import always
#    reads the patched source rather than stale bytecode.
import os, sys

# ── Patch configuration.py in the runtime copy ───────────────────────────
_CFG_PATH = '/content/HGRIA/backend/core/configuration.py'

with open(_CFG_PATH, 'r') as _f:
    _src = _f.read()

_OLD = '''    def __getattr__(self, name: str) -> Any:
        if name.startswith("_"):
            raise AttributeError(name)
        if name in self._data:
            val = self._data[name]
            if isinstance(val, dict):
                return _Namespace(val)
            return val
        raise AttributeError(f"No config section: {name}")'''

_NEW = '''    # Sections that must remain plain dicts (callers use dict methods on them).
    _PLAIN_DICT_SECTIONS = frozenset({"gesture_cooldowns_ms", "custom_gestures"})

    def __getattr__(self, name: str) -> Any:
        if name.startswith("_"):
            raise AttributeError(name)
        if name in self._data:
            val = self._data[name]
            if isinstance(val, dict) and name not in self._PLAIN_DICT_SECTIONS:
                return _Namespace(val)
            return val
        raise AttributeError(f"No config section: {name}")'''

if _OLD in _src:
    _src = _src.replace(_OLD, _NEW)
    with open(_CFG_PATH, 'w') as _f:
        _f.write(_src)
    print('\u2713 Patched configuration.py')
elif '_PLAIN_DICT_SECTIONS' in _src:
    print('\u2713 configuration.py already contains the fix — no patch needed')
else:
    print('\u26a0 Could not locate patch target in configuration.py — check manually')

# ── Evict cached modules ─────────────────────────────────────────────────
_PREFIXES = ('backend.', 'backend', 'dynamic_gestures.')
_evicted = [k for k in list(sys.modules) if k.startswith(_PREFIXES)]
for _mod in _evicted:
    del sys.modules[_mod]

print(f'\u2713 Evicted {len(_evicted)} cached module(s) — fresh import guaranteed')

In [ ]:
# Step 7: Start the HGRIA Server (blocking)
#
# The server opens the ngrok tunnel internally, then prints:
#   Public URL : https://xxxx.ngrok-free.app
#   Frontend   : https://qtannguyen-researcher.github.io/HGRIA/?server=...
#
# Copy the Frontend URL and open it in your browser.
# Press the Stop button (\u25a0) to terminate the server.
from backend.main import SystemOrchestrator

print("Starting HGRIA Backend Server...")
print("The public ngrok URL will appear below once the server is ready.")
print("-" * 60)

orchestrator = SystemOrchestrator(config_path)
orchestrator.start()

## Post-Launch Instructions

### Accessing the Frontend

After Step 7 starts, look for the `Frontend :` line in the output and open that URL directly.
It looks like:
```
Frontend   : https://qtannguyen-researcher.github.io/HGRIA/?server=https://xxxx.ngrok-free.app
```

### If ngrok URL Changes

1. Stop the server (interrupt Step 7)
2. Re-run Steps 6 and 7
3. Use the new `Frontend :` URL printed in the output

### Troubleshooting

| Issue | Solution |
|-------|----------|
| Colab session timeout | Re-run Step 7 (server restart is fast) |
| ngrok URL changed | Re-run Steps 6 and 7, use new Frontend URL |
| `authtoken does not look valid` | Edit `NGROK_AUTHTOKEN` in Step 5 with a real token |
| Code changes not taking effect | Re-run Steps 4–7 (Step 4 always does a fresh copy) |
| Webcam denied | Use keyboard fallback (Arrow keys, Space, P, S) |
| High latency | Check Colab GPU availability (Runtime > Change runtime type) |
| Drive not mounted | Re-run Step 3 |

### Keyboard Controls (Fallback)
- Arrow Keys: Move
- Space: Jump
- P: Pause
- S: Speed Boost
- Enter: Confirm